# PI Data Science · Track & Field — análise de vendas

Notebook único da entrega final: **dicionário de dados → limpeza → estatística descritiva → EDA →
inferência → OLAP → Machine Learning → previsão → insights**.

**Como executar (do início ao fim, sem arquivos externos):**

| Ambiente | Passos |
|---|---|
| Google Colab | *Ambiente de execução → Alterar o tipo → **R***. Depois *Executar tudo*. Os módulos R são baixados do GitHub. |
| Jupyter local | Instale o IRkernel e abra este arquivo na pasta `notebooks/` do clone. Usa os arquivos de `R/`. |
| RStudio / VS Code | Abra o `.ipynb` com kernel R, a partir da pasta `notebooks/`. |

A base é **gerada pela semente fixa** (`SEMENTE = 42`); nenhum CSV precisa existir.
Opcionalmente, `USAR_SPARK <- TRUE` lê a última publicação do pipeline Spark/HDFS do mesmo clone.
Tempo esperado: 1–3 minutos (inclui instalar `randomForest`, se faltar).

> **Dados sintéticos.** Os padrões da base são hipóteses do simulador (seção 2). Os resultados demonstram o
> método; não representam a operação real da Track & Field.

In [ ]:
options(repr.plot.width = 11, repr.plot.height = 5, width = 140, warn = 1)
SEMENTE    <- 42L
N_VENDAS   <- 5000L
USAR_SPARK <- FALSE  # TRUE: usa results/bigdata/ultima.txt, se o pipeline Spark já rodou neste clone
REPO_RAW   <- "https://raw.githubusercontent.com/luisfeob34/pi-data-science-track-field/main"

pacotes <- c("ggplot2", "randomForest")
faltantes <- pacotes[!vapply(pacotes, requireNamespace, logical(1), quietly = TRUE)]
if (length(faltantes)) install.packages(faltantes, repos = "https://cloud.r-project.org")
library(ggplot2)

# Usa os módulos do clone; fora dele (ex.: Colab), baixa os mesmos arquivos do GitHub.
modulos <- c("dados", "lotes", "estatistica", "previsao", "visualizacao")
candidatas <- c(".", "..")
raiz <- candidatas[file.exists(file.path(candidatas, "R", "dados.R"))][1]
if (is.na(raiz)) {
  raiz <- file.path(tempdir(), "pi-track-field")
  dir.create(file.path(raiz, "R"), recursive = TRUE, showWarnings = FALSE)
  for (m in modulos)
    download.file(sprintf("%s/R/%s.R", REPO_RAW, m), file.path(raiz, "R", paste0(m, ".R")), quiet = TRUE, mode = "wb")
  cat("Módulos R baixados do GitHub.\n")
}
raiz <- normalizePath(raiz)
for (m in modulos) source(file.path(raiz, "R", paste0(m, ".R")), encoding = "UTF-8")
cat("Raiz do projeto:", raiz, "\n", R.version.string, "\n")

In [ ]:
# Formatação e tema dos gráficos
num   <- function(x, d = 0) formatC(x, format = "f", digits = d, big.mark = ".", decimal.mark = ",")
reais <- function(x) paste0("R$ ", num(x, 2))
pct   <- function(x, d = 1) paste0(formatC(100 * x, format = "f", digits = d, decimal.mark = ","), "%")
tema  <- theme_minimal(base_size = 12) +
  theme(plot.title = element_text(face = "bold"), panel.grid.minor = element_blank())
eixo_reais <- scale_y_continuous(labels = function(v) num(v))

# Agregação usada em toda a análise: um "cubo" com as medidas do modelo dimensional.
resumir <- function(x, dims) {
  medidas <- data.frame(Pedidos = rep(1L, nrow(x)), Itens = x$Quantidade,
                        Faturamento = x[["Valor Total"]], Desconto_R = x[["Valor Desconto"]])
  r <- aggregate(medidas, by = x[dims], FUN = sum)
  r$Ticket <- r$Faturamento / r$Pedidos
  r$Participacao <- r$Faturamento / sum(r$Faturamento)
  r <- r[order(-r$Faturamento), ]
  rownames(r) <- NULL
  r
}

## 1. Situação-problema

Uma varejista de artigos esportivos vende em **loja física, e-commerce e marketplace**, nas cinco regiões.
Ela precisa decidir **onde concentrar estoque, mix de canal, calendário de campanhas e política de desconto**.
Hoje as vendas chegam em lotes de CSV, sem visão consolidada.

| # | Pergunta de negócio | Onde é respondida |
|---|---|---|
| P1 | Quais produtos e categorias concentram o faturamento? | EDA 6.1, OLAP drill-down |
| P2 | Qual canal vende melhor cada categoria, e isso muda por região? | EDA 6.3, OLAP slice e pivot |
| P3 | Quando ocorrem os picos e vales de venda? | EDA 6.2 e 6.6, OLAP roll-up, previsão |
| P4 | O desconto aumenta os itens por pedido o bastante para compensar? | EDA 6.5, ML 9.5 |
| P5 | É possível prever os itens de um pedido para dimensionar campanhas e estoque? | ML 9 |

Relação com o **ODS 8** (trabalho decente e crescimento econômico): decisões comerciais baseadas em dados
para pequenas e médias operações de varejo.

## 2. Identificação da base

| Item | Descrição |
|---|---|
| Origem | Simulador `R/dados.R::gerar_vendas`, executado na VM Ubuntu (P1) e processado no pipeline Spark/HDFS |
| Grão | **1 linha = 1 pedido de 1 produto** em um lote (`ID Execucao`) |
| Período | 01/01/2026 a 31/12/2026 |
| Volume neste notebook | `N_VENDAS` pedidos (padrão 5.000), semente `SEMENTE` |
| Licença / privacidade | Dados sintéticos. `Cliente` é um código fictício, sem dado pessoal |
| Documentação | Dicionário na seção 3 e em `docs/dicionario-dados.md` |

**Hipóteses do simulador** (padrões plantados para que a análise tenha o que descobrir):
sazonalidade mensal com picos em maio, novembro e dezembro e crescimento de 15% no ano; produtos sazonais
(jaqueta no inverno, garrafa e shorts no verão); canal preferido diferente por categoria; peso das regiões;
mais desconto no marketplace e na Black Friday; itens por pedido crescentes com o desconto.
A tabela abaixo mostra os parâmetros exatos.

In [ ]:
if (USAR_SPARK) {
  spark <- ultima_base_spark(raiz)
  if (is.null(spark)) stop("Nenhuma publicação Spark encontrada. Rode o pipeline ou use USAR_SPARK <- FALSE.")
  brutos <- ler_vendas(spark$arquivo)
  origem <- paste("Pipeline Spark:", spark$id)
} else {
  brutos <- gerar_vendas(N_VENDAS, SEMENTE)
  origem <- sprintf("Simulador R (n = %s, semente = %d)", num(N_VENDAS), SEMENTE)
}
cat("Origem:", origem, "\nRegistros brutos:", num(nrow(brutos)), "\n\n")
str(brutos)
head(brutos)

In [ ]:
# Parâmetros do simulador (documentação das hipóteses; não são medidas reais)
perfil_simulador$produtos
cat("\nCanal por categoria (probabilidades):\n"); print(perfil_simulador$canal_por_categoria)
cat("\nPeso por região:\n"); print(setNames(perfil_simulador$peso_regiao, perfil_simulador$regioes))
cat("\nSazonalidade mensal do volume:\n"); print(setNames(perfil_simulador$sazonalidade_mes, month.abb))

## 3. Dicionário de dados

**Campos de origem** (CSV gerado pelo simulador ou recebido no lote):

| Campo | Tipo | Descrição | Domínio / regra de validação |
|---|---|---|---|
| ID Pedido | inteiro | Número do pedido no lote | ≥ 1; único por `ID Execucao + ID Pedido` |
| Data da Venda | data `AAAA-MM-DD` | Data do pedido | data de calendário válida |
| Cliente | texto | Código sintético do cliente | `Cliente_00001`…; não identifica pessoa e repete entre lotes |
| Produto | categórico | Produto vendido | 10 produtos |
| Categoria | categórico | Grupo do produto | Vestuário, Calçados, Acessórios |
| Canal de Venda | categórico | Canal do pedido | Loja Física, E-commerce, Marketplace |
| Região | categórico | Região do Brasil | Norte, Nordeste, Centro-Oeste, Sudeste, Sul |
| Preço Unitário | decimal (R$) | Preço de tabela | > 0 |
| Quantidade | inteiro | Itens do pedido | inteiro > 0 (simulador: 1 a 5) |
| Desconto | decimal (fração) | Desconto sobre o valor bruto | 0 a 1 (simulador: 0; 0,05; 0,10; 0,15) |
| Valor Total | decimal (R$) | Valor líquido do pedido | ≥ 0; **recalculado** no tratamento |
| ID Execucao | texto | Lote que gerou o registro | `exec-…`; `avulso` quando ausente |
| Gerado em UTC | data-hora | Momento da geração do lote | só lotes da VM; não segue para a análise |

**Campos derivados** (`tratar_vendas`, mesmas regras no Spark):

| Campo | Regra |
|---|---|
| Valor Bruto | Preço Unitário × Quantidade, em centavos (empate para o par) |
| Valor Desconto | Valor Bruto × Desconto, em centavos |
| Valor Total | Valor Bruto − Valor Desconto (substitui o valor recebido) |
| Possui Desconto | Sim / Não |
| Desconto Percentual | 100 × Desconto |
| Faixa de Desconto | Sem desconto; Até 5%; De 6% a 10%; Acima de 10% |
| Valor por Item | Valor Total ÷ Quantidade |
| Porte do Pedido | Unitário (1); Pequeno (2–3); Grande (4+) |
| Faixa de Valor | Até R\$ 249,99; R\$ 250 a 499,99; R\$ 500 a 999,99; R\$ 1.000 ou mais |
| Ano, Número do Mês, Mês, Periodo, Trimestre, Dia da Semana | Atributos de calendário da Data da Venda |

No Data Lake, a camada gold organiza esses campos em `fato_vendas` e nas dimensões produto, canal, região e data
(`docs/arquitetura-pi.md`).

## 4. Limpeza e qualidade dos dados

Regras de `tratar_vendas` (as mesmas do Spark, conferidas linha a linha nos testes de integração):

1. Aparar espaços e remover **duplicatas exatas**.
2. Remover registros com **campos ausentes**, **datas inválidas** ou **números não finitos**.
3. Remover valores **fora do domínio** (preço ≤ 0, quantidade ≤ 0 ou fracionária, desconto fora de 0–1, total < 0).
4. Descartar **IDs conflitantes** (mesmo lote e pedido com conteúdos diferentes): não se escolhe um arbitrariamente.
5. **Recalcular** o valor total em centavos; a divergência é contada, não ignorada.
6. Valores atípicos **válidos** não são removidos: são vendas reais do negócio.

A base gerada vem íntegra. Para demonstrar as regras, a célula abaixo injeta defeitos controlados em uma cópia.

In [ ]:
sujo <- data.frame(lapply(brutos, as.character), check.names = FALSE)  # como lido de um CSV
defeitos <- sujo[1:6, ]
defeitos$Quantidade[1]          <- "0"           # fora do domínio
defeitos[["Data da Venda"]][2]  <- "2026-02-30"  # data inexistente
defeitos[["Preço Unitário"]][3] <- "abc"         # número inválido
defeitos$Cliente[4]             <- ""            # ausente
defeitos$Desconto[5]            <- "1.5"         # desconto acima de 100%
# a linha 6 fica idêntica à original: duplicata exata
conflito <- sujo[7, ]; conflito$Produto <- "Produto Conflitante"  # mesmo ID, conteúdo diferente
sujo[["Valor Total"]][8] <- "1"                   # total errado: será recalculado
sujo <- rbind(sujo, defeitos, conflito)

demo    <- tratar_vendas(sujo)
tratado <- tratar_vendas(brutos)
base    <- tratado$dados
data.frame(Etapa = tratado$auditoria$Etapa,
           `Base analisada` = tratado$auditoria$Registros,
           `Cópia com defeitos` = demo$auditoria$Registros, check.names = FALSE)

In [ ]:
cat("Valores ausentes após o tratamento:", sum(is.na(base)), "\n")
documentados <- c(colunas_vendas, "ID Execucao", "Valor Bruto", "Valor Desconto", "Possui Desconto",
  "Desconto Percentual", "Faixa de Desconto", "Valor por Item", "Porte do Pedido", "Faixa de Valor",
  "Ano", "Número do Mês", "Mês", "Periodo", "Trimestre", "Dia da Semana")
cat("Campos sem documentação:", length(setdiff(names(base), documentados)),
    "| Documentados ausentes:", length(setdiff(documentados, names(base))), "\n")
cat("Tipos:\n"); print(vapply(base, function(v) class(v)[1], character(1)))

# Atípicos pela regra de Tukey (1,5 × IQR): contados, não removidos
vt <- base[["Valor Total"]]
lim <- quantile(vt, .75) + 1.5 * IQR(vt)
cat(sprintf("\nPedidos acima de %s (atípicos de Tukey): %s (%s). Todos são de: %s.\n", reais(lim),
    num(sum(vt > lim)), pct(mean(vt > lim)), paste(sort(unique(base$Categoria[vt > lim])), collapse = ", ")))

**Leitura.** A coluna *Cópia com defeitos* mostra cada defeito capturado pela regra correspondente:
1 duplicata, 1 ausente, 1 data inválida, 1 número inválido, 2 fora do domínio, 2 registros do ID conflitante
(o original e a cópia) e 1 total recalculado. A base analisada não perde registros.
Os atípicos de valor são pedidos legítimos de itens caros em maior quantidade: ficam na base e são tratados com
estatísticas robustas (mediana, quartis, testes não paramétricos).

## 5. Estatística descritiva

In [ ]:
data.frame(
  Indicador = c("Pedidos", "Itens vendidos", "Faturamento líquido", "Ticket médio", "Ticket mediano",
                "Desconto concedido", "Pedidos com desconto", "Itens por pedido"),
  Valor = c(num(nrow(base)), num(sum(base$Quantidade)), reais(sum(vt)), reais(mean(vt)), reais(median(vt)),
            reais(sum(base[["Valor Desconto"]])), pct(mean(base$Desconto > 0)), num(mean(base$Quantidade), 2)))

In [ ]:
descr <- descrever_vendas(base)
numericas <- vapply(descr, is.numeric, logical(1))
descr[numericas] <- lapply(descr[numericas], round, 2)
descr

In [ ]:
freq <- frequencias_vendas(base); freq$Proporcao <- pct(freq$Proporcao); freq
prob <- probabilidades_vendas(base); prob$Probabilidade_empirica <- pct(prob$Probabilidade_empirica); prob
cat("\nCorrelação de Spearman entre as variáveis numéricas:\n")
round(cor(base[c("Preço Unitário", "Quantidade", "Desconto", "Valor Total")], method = "spearman"), 2)

In [ ]:
q <- base$Quantidade
cat(sprintf("Valor do pedido: média %s, mediana %s → %s.\n", reais(mean(vt)), reais(median(vt)),
  if (mean(vt) > median(vt)) "assimetria à direita: poucos pedidos grandes puxam a média" else "sem assimetria à direita"))
cat(sprintf("Coeficiente de variação do valor: %s → ticket heterogêneo; compare grupos por mediana e quartis.\n",
  pct(sd(vt) / mean(vt))))
cat(sprintf("Itens por pedido: média %s, moda %s; %s dos pedidos têm 1 item.\n",
  num(mean(q), 2), moda_descritiva(q), pct(mean(q == 1))))
cat(sprintf("Correlação desconto × quantidade (Spearman): %s.\n",
  num(cor(base$Desconto, q, method = "spearman"), 2)))

**Leitura.** O valor do pedido é assimétrico e heterogêneo porque mistura meias de R\$ 49,90 com tênis de
R\$ 499,90. Por isso a média sozinha engana: as comparações entre grupos usam mediana, quartis e testes
não paramétricos. A correlação positiva entre desconto e quantidade é a primeira pista para P4, mas correlação
não separa o efeito do desconto do efeito da categoria — isso é feito no modelo da seção 9.

## 6. Análise exploratória

### 6.1 Faturamento por produto e categoria (P1)

In [ ]:
por_produto <- resumir(base, c("Produto", "Categoria"))
ggplot(por_produto, aes(reorder(Produto, Faturamento), Faturamento, fill = Categoria)) +
  geom_col(width = .7) + coord_flip() + eixo_reais +
  labs(x = NULL, y = "Faturamento (R$)", title = "Faturamento por produto") + tema

por_categoria <- resumir(base, "Categoria")
top3 <- head(por_produto, 3)
cat(sprintf("%s lidera com %s do faturamento; %s tem o maior ticket médio (%s).\n",
  por_categoria$Categoria[1], pct(por_categoria$Participacao[1]),
  por_categoria$Categoria[which.max(por_categoria$Ticket)], reais(max(por_categoria$Ticket))))
cat(sprintf("Os três produtos líderes (%s) somam %s do faturamento.\n",
  paste(top3$Produto, collapse = ", "), pct(sum(top3$Participacao))))
por_categoria[c("Categoria", "Pedidos", "Itens", "Faturamento", "Ticket", "Participacao")]

### 6.2 Evolução mensal (P3)

In [ ]:
mensal <- resumir(base, "Periodo")
mensal <- mensal[order(mensal$Periodo), ]
mensal$Data <- as.Date(paste0(mensal$Periodo, "-01"))
mensal$Indice <- mensal$Faturamento / mean(mensal$Faturamento)
ggplot(mensal, aes(Data, Faturamento)) +
  geom_col(fill = "#1f6c5b", width = 20) +
  geom_hline(yintercept = mean(mensal$Faturamento), linetype = 2) +
  scale_x_date(date_labels = "%m/%y", date_breaks = "1 month") + eixo_reais +
  labs(x = NULL, y = "Faturamento (R$)", title = "Faturamento mensal (linha = média)") + tema

pico <- mensal[which.max(mensal$Indice), ]; vale <- mensal[which.min(mensal$Indice), ]
cat(sprintf("Pico: %s (%s acima da média). Vale: %s (%s abaixo da média).\n",
  pico$Periodo, pct(pico$Indice - 1), vale$Periodo, pct(1 - vale$Indice)))
mensal[c("Periodo", "Pedidos", "Faturamento", "Indice")]

### 6.3 Canal preferido por categoria (P2)

In [ ]:
cc <- resumir(base, c("Categoria", "Canal de Venda"))
cc$Share <- cc$Faturamento / ave(cc$Faturamento, cc$Categoria, FUN = sum)
ggplot(cc, aes(`Canal de Venda`, Categoria, fill = Share)) +
  geom_tile(colour = "white") + geom_text(aes(label = pct(Share, 0))) +
  scale_fill_gradient(low = "#eef5f2", high = "#1f6c5b", labels = function(v) pct(v, 0)) +
  labs(x = NULL, y = NULL, fill = "% da categoria", title = "Participação de cada canal no faturamento da categoria") + tema

dominante <- do.call(rbind, lapply(split(cc, cc$Categoria), function(z) z[which.max(z$Share), ]))
for (i in seq_len(nrow(dominante)))
  cat(sprintf("%s: canal principal %s (%s do faturamento da categoria).\n",
    dominante$Categoria[i], dominante[["Canal de Venda"]][i], pct(dominante$Share[i])))

### 6.4 Distribuição do valor por categoria e por região

In [ ]:
caixa <- rbind(
  data.frame(Dimensao = "Categoria", Grupo = base$Categoria, Valor = vt),
  data.frame(Dimensao = "Região", Grupo = base[["Região"]], Valor = vt))
ggplot(caixa, aes(Grupo, Valor)) +
  geom_boxplot(fill = "#c8ded4", colour = "#1f6c5b", outlier.alpha = .3) +
  facet_wrap(~Dimensao, scales = "free_x") + scale_y_log10(labels = function(v) num(v)) +
  labs(x = NULL, y = "Valor do pedido (R$, escala log)", title = "Valor do pedido") + tema
aggregate(Valor ~ Dimensao + Grupo, caixa, median)

**Leitura.** Entre categorias, as caixas ficam em patamares diferentes (Calçados acima). Entre regiões, as caixas
se sobrepõem: a região muda o **volume** de pedidos, não o valor típico de cada pedido. A seção 7 testa isso formalmente.

### 6.5 Desconto × itens por pedido (P4)

In [ ]:
dq <- aggregate(Quantidade ~ Desconto + Categoria, base, mean)
ggplot(dq, aes(factor(100 * Desconto), Quantidade, colour = Categoria, group = Categoria)) +
  geom_line(linewidth = 1) + geom_point(size = 2.5) +
  labs(x = "Desconto (%)", y = "Itens por pedido (média)", title = "Itens por pedido conforme o desconto") + tema
dc <- aggregate(Desconto ~ `Canal de Venda`, base, mean); dc$Desconto <- pct(dc$Desconto); dc

**Leitura.** Em todas as categorias, pedidos com mais desconto têm mais itens. O desconto médio também varia por
canal, e o canal varia por categoria: as variáveis se confundem. O modelo da seção 9 separa esses efeitos.

### 6.6 Sazonalidade por produto (P3)

In [ ]:
pm <- resumir(base, c("Número do Mês", "Produto"))
pm$Indice <- pm$Pedidos / ave(pm$Pedidos, pm$Produto, FUN = mean)
sazonais <- c("Jaqueta Corta-Vento", "Garrafa Térmica", "Shorts Running", "Tênis Running")
ggplot(subset(pm, Produto %in% sazonais), aes(`Número do Mês`, Indice, colour = Produto)) +
  geom_line(linewidth = 1) + geom_point() + geom_hline(yintercept = 1, linetype = 2) +
  scale_x_continuous(breaks = 1:12) +
  labs(x = "Mês", y = "Pedidos ÷ média mensal do produto", title = "Índice sazonal de produtos selecionados") + tema
picos_sazonais <- vapply(sazonais, function(p) {
  z <- pm[pm$Produto == p, ]
  sprintf("%s no mês %d (índice %s)", p, z[["Número do Mês"]][which.max(z$Indice)], num(max(z$Indice), 2))
}, character(1))
cat(paste("Pico:", picos_sazonais), sep = "\n")

## 7. Testes estatísticos

Kruskal-Wallis (valor do pedido entre grupos), Spearman (desconto × valor) e qui-quadrado (categoria × canal),
com ajuste de Holm a 5%. Detalhes e premissas em `docs/metodologia-r.md`.

In [ ]:
testes <- testes_estatisticos(base)
testes$Efeito <- round(testes$Efeito, 3)
testes$p_ajustado <- signif(testes$p_ajustado, 3)
testes[c("Teste", "Efeito", "Medida", "p_ajustado", "Conclusao")]

**Leitura.** Um p-valor pequeno indica que a diferença dificilmente é acaso; o **efeito** (epsilon², rho, V de Cramér)
diz se ela é grande o bastante para importar. Associação não comprova causa.

## 8. Operações OLAP

O cubo tem as medidas da `fato_vendas` (pedidos, itens, faturamento, desconto) e as dimensões
**produto/categoria, canal, região e tempo** (dia → mês → trimestre → ano). Cada operação responde a uma pergunta.

In [ ]:
cubo <- resumir(base, c("Categoria", "Canal de Venda", "Região", "Trimestre"))
cat("Células do cubo Categoria × Canal × Região × Trimestre:", nrow(cubo), "\n")
head(cubo, 8)

**Roll-up (P3):** mês → trimestre → ano. A soma se conserva em cada nível.

In [ ]:
por_trimestre <- resumir(base, "Trimestre"); por_trimestre <- por_trimestre[order(por_trimestre$Trimestre), ]
por_ano <- resumir(base, "Ano")
stopifnot(isTRUE(all.equal(sum(mensal$Faturamento), sum(por_trimestre$Faturamento))),
          isTRUE(all.equal(sum(por_trimestre$Faturamento), sum(por_ano$Faturamento))))
por_trimestre[c("Trimestre", "Pedidos", "Faturamento", "Participacao")]
por_ano[c("Ano", "Pedidos", "Faturamento")]

**Drill-down (P1):** do trimestre mais forte para categoria e produto.

In [ ]:
t_top <- por_trimestre$Trimestre[which.max(por_trimestre$Faturamento)]
cat("Trimestre de maior faturamento:", t_top, "\n")
head(resumir(base[base$Trimestre == t_top, ], c("Categoria", "Produto")), 10)

**Slice (P2):** fixa o canal E-commerce e compara categorias.

In [ ]:
resumir(base[base[["Canal de Venda"]] == "E-commerce", ], "Categoria")

**Dice (P1 + P3):** subcubo Calçados × {Sudeste, Sul} × 4º trimestre — alvo de uma campanha de fim de ano.

In [ ]:
dado <- base[base$Categoria == "Calçados" & base[["Região"]] %in% c("Sudeste", "Sul") & grepl("-T4$", base$Trimestre), ]
resumir(dado, c("Produto", "Região"))

**Pivot (P2):** região nas linhas, canal nas colunas (% do faturamento da região).

In [ ]:
pivo <- xtabs(`Valor Total` ~ `Região` + `Canal de Venda`, data = base)
round(100 * prop.table(pivo, 1), 1)

No pipeline, as mesmas operações rodam no Spark SQL sobre a camada gold. Exemplo equivalente ao roll-up:

```sql
SELECT year(data_venda) AS ano, quarter(data_venda) AS trimestre, month(data_venda) AS mes,
       count(*) AS pedidos, sum(valor_total) AS faturamento
FROM vendas
GROUP BY ROLLUP(year(data_venda), quarter(data_venda), month(data_venda))
```

## 9. Machine Learning — prever itens por pedido (P4 e P5)

**Tarefa:** regressão da `Quantidade` (itens por pedido, contagem de 1 a 5).
**Por que essa variável:** é o que o negócio controla com campanhas e desconto e o que dimensiona estoque.
O valor do pedido seria trivial de "prever" porque é calculado a partir de preço, quantidade e desconto.

**Variáveis explicativas:** categoria, produto, canal, região, desconto, mês e dia da semana.
**Excluídas por vazamento:** Valor Total, Valor Bruto, Valor Desconto, Valor por Item, Porte do Pedido e
Faixa de Valor — todas derivadas da própria quantidade.

**Modelos comparados:**

| Modelo | Por que entra |
|---|---|
| Média geral | linha de base: qualquer modelo precisa superá-la |
| Média por categoria | linha de base forte e simples |
| GLM Poisson (itens extras) | modelo clássico para contagens; coeficientes interpretáveis |
| Random Forest | não linear, captura interações sem especificá-las |

**Avaliação:** 80% treino / 20% teste. No treino, validação cruzada de 5 dobras (MAE, RMSE, R²).
Seleção pela **regra de 1 desvio-padrão**: entre os modelos cujo RMSE médio fica a até 1 DP do melhor, escolhe-se
o mais simples. O teste é usado **uma única vez**, no fim, para estimar o erro do modelo escolhido.

In [ ]:
ml <- droplevels(data.frame(
  Quantidade = base$Quantidade,
  Categoria  = factor(base$Categoria),
  Produto    = factor(base$Produto),
  Canal      = factor(base[["Canal de Venda"]]),
  Regiao     = factor(base[["Região"]]),
  Desconto   = base$Desconto,
  Mes        = factor(base[["Número do Mês"]], levels = 1:12),
  DiaSemana  = factor(base[["Dia da Semana"]])))
preco <- base[["Preço Unitário"]]

set.seed(SEMENTE)
idx_teste <- sample(nrow(ml), round(.2 * nrow(ml)))
treino <- ml[-idx_teste, ]; teste <- ml[idx_teste, ]
cat("Treino:", num(nrow(treino)), "| Teste:", num(nrow(teste)), "\n")

metricas <- function(y, p) c(MAE = mean(abs(y - p)), RMSE = sqrt(mean((y - p)^2)),
                             R2 = 1 - sum((y - p)^2) / sum((y - mean(y))^2))
formula_glm <- I(Quantidade - 1) ~ Categoria + Desconto + Canal + Regiao + Mes
# Cada entrada ajusta no treino e devolve uma função de previsão.
modelos <- list(
  "Média geral" = function(tr) { m <- mean(tr$Quantidade); function(te) rep(m, nrow(te)) },
  "Média por categoria" = function(tr) {
    m <- tapply(tr$Quantidade, tr$Categoria, mean); function(te) unname(m[as.character(te$Categoria)]) },
  "GLM Poisson" = function(tr) {
    f <- glm(formula_glm, family = poisson, data = tr); function(te) pmin(5, 1 + predict(f, te, type = "response")) },
  "Random Forest" = function(tr) {
    # Aviso "cinco ou menos valores" é esperado: a regressão da contagem é intencional.
    f <- suppressWarnings(randomForest::randomForest(Quantidade ~ ., data = tr, ntree = 300, nodesize = 20))
    function(te) unname(predict(f, te)) })
simplicidade <- names(modelos)  # ordem do mais simples para o mais complexo

### 9.1 Validação cruzada (somente treino)

In [ ]:
set.seed(SEMENTE)
dobras <- sample(rep(1:5, length.out = nrow(treino)))
cv <- do.call(rbind, lapply(names(modelos), function(nome) {
  r <- t(sapply(1:5, function(k)
    metricas(treino$Quantidade[dobras == k], modelos[[nome]](treino[dobras != k, ])(treino[dobras == k, ]))))
  data.frame(Modelo = nome, MAE = mean(r[, "MAE"]), RMSE = mean(r[, "RMSE"]), RMSE_dp = sd(r[, "RMSE"]),
             R2 = mean(r[, "R2"]))
}))
melhor <- which.min(cv$RMSE)
candidatos <- cv$Modelo[cv$RMSE <= cv$RMSE[melhor] + cv$RMSE_dp[melhor]]
escolhido <- simplicidade[simplicidade %in% candidatos][1]
cv[-1] <- lapply(cv[-1], round, 4)
cv
cat(sprintf("Menor RMSE: %s. Escolhido pela regra de 1 DP: %s.\n", cv$Modelo[melhor], escolhido))

### 9.2 Avaliação final no conjunto de teste

In [ ]:
set.seed(SEMENTE)
preditores <- lapply(modelos, function(ajustar) ajustar(treino))  # ajuste final: treino completo
prever <- preditores[[escolhido]]
final <- do.call(rbind, lapply(names(preditores), function(nome)
  data.frame(Modelo = nome, t(metricas(teste$Quantidade, preditores[[nome]](teste))))))
rmse_base <- final$RMSE[final$Modelo == "Média geral"]
final$Ganho_RMSE <- pct(1 - final$RMSE / rmse_base)
final[c("MAE", "RMSE", "R2")] <- lapply(final[c("MAE", "RMSE", "R2")], round, 4)
final
f_escolhido <- final[final$Modelo == escolhido, ]
cat(sprintf("Modelo escolhido (%s) no teste: MAE %s itens, RMSE %s, R² %s; RMSE %s menor que a média geral.\n",
  escolhido, num(f_escolhido$MAE, 2), num(f_escolhido$RMSE, 2), num(f_escolhido$R2, 2), f_escolhido$Ganho_RMSE))

In [ ]:
# Calibração: média prevista × observada por categoria e desconto (teste)
cal <- data.frame(Categoria = teste$Categoria, Desconto = teste$Desconto,
                  Observado = teste$Quantidade, Previsto = prever(teste))
calib <- aggregate(cbind(Observado, Previsto) ~ Categoria + Desconto, cal, mean)
ggplot(calib, aes(Previsto, Observado, colour = Categoria)) +
  geom_abline(linetype = 2) + geom_point(aes(shape = factor(100 * Desconto)), size = 3) +
  labs(shape = "Desconto (%)", title = paste("Calibração no teste —", escolhido, "(linha = previsão perfeita)")) + tema

### 9.3 O que explica os itens por pedido

In [ ]:
set.seed(SEMENTE)
rf <- suppressWarnings(randomForest::randomForest(Quantidade ~ ., data = treino, ntree = 500, nodesize = 20, importance = TRUE))
imp <- randomForest::importance(rf, type = 1)
imp <- data.frame(Variavel = rownames(imp), Importancia = imp[, 1])
ggplot(imp, aes(reorder(Variavel, Importancia), Importancia)) + geom_col(fill = "#1f6c5b") + coord_flip() +
  labs(x = NULL, y = "Aumento do erro ao embaralhar a variável (%IncMSE)", title = "Importância — Random Forest") + tema

glm_final <- glm(formula_glm, family = poisson, data = treino)
drop1(glm_final, test = "LRT")

In [ ]:
b <- coef(summary(glm_final))["Desconto", ]
rr <- exp(.05 * (b[["Estimate"]] + c(0, -1.96, 1.96) * b[["Std. Error"]]))
cat(sprintf("Cada +5 p.p. de desconto multiplica a média de itens extras por %s (IC95%% %s a %s).\n",
  num(rr[1], 2), num(rr[2], 2), num(rr[3], 2)))
lrt <- drop1(glm_final, test = "LRT")
p_lrt <- setNames(lrt[["Pr(>Chi)"]], rownames(lrt))[-1]
cat("Variáveis com efeito significativo (LRT, 5%):", paste(names(p_lrt)[p_lrt < .05], collapse = ", "), "\n")
cat("Sem evidência de efeito:", paste(names(p_lrt)[p_lrt >= .05], collapse = ", "), "\n")

**Leitura.** Importância e teste de razão de verossimilhança apontam o que de fato move os itens por pedido.
Variáveis sem efeito, dado o restante, não devem orientar decisões de quantidade — mesmo que apareçam
"diferentes" em um gráfico isolado (efeito de confusão, ver 6.5).

**Nota de honestidade metodológica.** O simulador gera a quantidade por uma regra de Poisson log-linear em
categoria e desconto. Por isso o GLM está corretamente especificado *aqui*, e é esperado que empate com a Random
Forest. Em dados reais a forma da relação é desconhecida: a comparação com um modelo flexível é o que protege
contra um modelo simples mal especificado.

### 9.4 Previsão de faturamento mensal (complemento)

Modelos de série temporal do painel (média, último mês, tendência), avaliados em origens móveis de um passo.
Com 12 meses não há ciclo anual suficiente para estimar sazonalidade: o pico de novembro/dezembro aparece
como erro, não como padrão aprendido.

In [ ]:
serie <- serie_mensal(base)
prev  <- prever_vendas(serie, 3)
cat(prev$motivo, "\n")
if (prev$disponivel) {
  print(prev$validacao)
  cat("\n", prev$explicacao, "\n\n")
  print(prev$previsao)
  print(grafico_previsao(prev))
}

### 9.5 Uso do modelo: simulação de política de desconto (P4)

Para cada categoria, aplica-se o mesmo desconto a todos os pedidos do teste e o modelo escolhido estima itens e
receita por pedido. **Receita não é lucro:** a base não tem custo nem margem.

In [ ]:
cenarios <- do.call(rbind, lapply(perfil_simulador$descontos, function(d) {
  cf <- teste; cf$Desconto <- d
  itens <- prever(cf)
  data.frame(Desconto = d, Categoria = cf$Categoria, Itens = itens, Receita = preco[idx_teste] * itens * (1 - d))
}))
politica <- aggregate(cbind(Itens, Receita) ~ Desconto + Categoria, cenarios, mean)
ref <- politica[politica$Desconto == 0, c("Categoria", "Receita")]; names(ref)[2] <- "Receita_sem_desconto"
politica <- merge(politica, ref)
politica$Variacao <- politica$Receita / politica$Receita_sem_desconto - 1
ggplot(politica, aes(factor(100 * Desconto), Variacao, colour = Categoria, group = Categoria)) +
  geom_hline(yintercept = 0, linetype = 2) + geom_line(linewidth = 1) + geom_point(size = 2.5) +
  scale_y_continuous(labels = function(v) pct(v, 0)) +
  labs(x = "Desconto aplicado (%)", y = "Receita por pedido vs. sem desconto",
       title = "Efeito estimado do desconto na receita por pedido") + tema
transform(politica[c("Categoria", "Desconto", "Itens", "Receita")],
          Itens = round(Itens, 2), Receita = reais(Receita), Variacao = pct(politica$Variacao))

**Leitura e limite causal.** No simulador, o desconto é sorteado por canal e mês, sem depender do cliente; por isso o
contrafactual é válido *nesta base*. Em dados reais o desconto é escolhido pela empresa (promoções em produtos
encalhados, clientes sensíveis a preço) e a estimativa ficaria enviesada. A recomendação é validar com
**teste A/B** antes de mudar a política, e incluir margem para avaliar lucro, não só receita.

## 10. Hadoop e Spark: por que no pipeline e não neste notebook

Este notebook processa alguns milhares de pedidos em memória: R basta. A escolha do **Spark (DataFrames e SQL)
sobre HDFS** vale para o fluxo de produção proposto, em que lotes chegam continuamente de muitos canais e lojas:

| V | No caso | O que o pipeline faz |
|---|---|---|
| Volume | histórico crescente de lotes | Parquet colunar no HDFS, processamento distribuível |
| Velocidade | lotes recorrentes | ingestão idempotente por hash, reprocessamento completo |
| Variedade | canais e origens distintas | esquema validado na bronze |
| Veracidade | duplicatas, conflitos, totais errados | quarentena com motivo; mesmas regras desta seção 4 |
| Valor | decisões das seções 6–9 | camada gold com fato e dimensões |

A demonstração usa um NameNode e um DataNode no mesmo host e Spark `local[2]`: integra os componentes, não prova
desempenho de cluster. Detalhes em `docs/arquitetura-pi.md` e `docs/validacao-bigdata.md`.

In [ ]:
spark_pub <- tryCatch(ultima_base_spark(raiz), error = function(e) NULL)
if (is.null(spark_pub)) {
  cat("Nenhuma publicação Spark neste ambiente (esperado no Colab). Rode scripts/pipeline_bigdata.sh no Ubuntu.\n")
} else {
  cat("Última publicação Spark:", spark_pub$id, "\n")
  print(read.csv(file.path(spark_pub$pasta, "resumo.csv"), check.names = FALSE))
  integridade <- file.path(spark_pub$pasta, "integridade_modelo.csv")
  if (file.exists(integridade)) print(read.csv(integridade, check.names = FALSE))
}

## 11. Insights para a situação-problema

In [ ]:
desc15 <- politica[politica$Desconto == max(politica$Desconto), ]
melhor_desc <- desc15[which.max(desc15$Variacao), ]
pior_desc   <- desc15[which.min(desc15$Variacao), ]
sem_efeito  <- names(p_lrt)[p_lrt >= .05]
insights <- c(
  sprintf("P1 · %s responde por %s do faturamento; %s tem o maior ticket (%s). Os três produtos líderes somam %s.",
    por_categoria$Categoria[1], pct(por_categoria$Participacao[1]),
    por_categoria$Categoria[which.max(por_categoria$Ticket)], reais(max(por_categoria$Ticket)), pct(sum(top3$Participacao))),
  sprintf("P2 · Canal principal: %s.",
    paste(sprintf("%s → %s (%s)", dominante$Categoria, dominante[["Canal de Venda"]], pct(dominante$Share, 0)), collapse = "; ")),
  sprintf("P3 · Pico em %s (%s acima da média) e vale em %s (%s abaixo). Picos sazonais: %s.",
    pico$Periodo, pct(pico$Indice - 1), vale$Periodo, pct(1 - vale$Indice), paste(picos_sazonais, collapse = "; ")),
  sprintf("P4 · +5 p.p. de desconto multiplicam os itens extras por %s. Com %s de desconto, a receita por pedido varia %s em %s e %s em %s (sem considerar margem).",
    num(rr[1], 2), pct(max(politica$Desconto), 0), pct(melhor_desc$Variacao), melhor_desc$Categoria,
    pct(pior_desc$Variacao), pior_desc$Categoria),
  sprintf("P5 · %s prevê itens por pedido com MAE de %s itens (R² %s), RMSE %s menor que a média geral. Sem efeito adicional (LRT): %s.",
    escolhido, num(f_escolhido$MAE, 2), num(f_escolhido$R2, 2), f_escolhido$Ganho_RMSE,
    if (length(sem_efeito)) paste(sem_efeito, collapse = ", ") else "nenhuma variável"))
cat(paste("•", insights), sep = "\n")

### Recomendações (condicionadas à validação com dados reais)

1. **Estoque e campanhas por calendário:** reforçar estoque antes de novembro e dezembro; antecipar jaquetas em maio
   e itens de verão em novembro. O índice sazonal (6.6) define quanto.
2. **Mix por canal:** priorizar no sortimento e na mídia de cada canal a categoria em que ele já é principal (6.3);
   região define volume (logística), não o tipo de pedido.
3. **Desconto seletivo:** concentrar descontos onde o ganho de itens compensa (categorias com pedidos de vários itens)
   e evitar descontos amplos onde quase todo pedido tem 1 item. Validar por teste A/B e incluir margem.
4. **Previsão operacional:** usar o modelo de itens por pedido para dimensionar campanhas; a previsão mensal só
   ganha sazonalidade com pelo menos dois anos de histórico.

## 12. Limitações

- Dados **sintéticos**: os padrões encontrados são os plantados no simulador; servem para validar o método.
- Sem custo/margem: as conclusões falam de receita, não de lucro.
- Um ano de histórico: sazonalidade anual não é estimável pela série; aparece apenas na EDA.
- `Cliente` não identifica recompra; não há análise de retenção.
- Efeito do desconto é causal apenas porque o simulador sorteia o desconto; em dados reais exige experimento.
- Infraestrutura Big Data em um único host: demonstra integração, não escala.

## 13. Conclusões

O fluxo completo — geração em VM, ingestão em lotes, tratamento com regras auditáveis, Data Lake em camadas com
modelo dimensional, OLAP, testes estatísticos e ML avaliado contra linhas de base — responde às cinco perguntas
de negócio com números rastreáveis. O próximo passo é substituir o simulador por dados reais do beneficiário e
repetir este notebook sem alterar o código.

In [ ]:
sessionInfo()